# Metaclonotype Clustering — Method Comparison

This notebook demonstrates and compares different metaclonotype clustering methods
available in mirpy:

| Method | Algorithm type | Paired support |
|---|---|---|
| `edit_distance` | Graph (Hamming/Levenshtein + Leiden/components) | Via chain-combine |
| `alice` | Enrichment-based seed neighbors | Via chain-combine |
| `tcrnet` | Enrichment-based seed neighbors | Via chain-combine |
| `tcrdist` | Continuous radius threshold | Via chain-combine |
| `tcremp` | Embedding + DBSCAN/OPTICS | Native paired embedding |
| `gliph` | k-mer token graph | Via chain-combine |

**Workflow:**
1. Load a benchmark TRB repertoire.
2. Compute clonotype diversity (baseline).
3. Cluster with each method → compute functional diversity.
4. Compare functional Hill curves across methods.
5. Load paired TRA/TRB data → compare TCREmp native paired vs edit-distance chain-combined.
6. Measure concordance between paired strategies.

---
**Required assets:** AIRR benchmark data (run `ensure_airr_benchmark()` once to download).

In [ ]:
# Cell 1: Environment versions
import platform, sys, importlib
print('Python', sys.version)
for pkg in ['mir', 'polars', 'numpy', 'sklearn', 'igraph']:
    try:
        m = importlib.import_module(pkg)
        print(f'{pkg} {getattr(m, "__version__", "?")}', end='  ')
    except ImportError:
        print(f'{pkg} NOT INSTALLED', end='  ')
print('\nPlatform', platform.machine())

In [ ]:
# Cell 2: Imports
import time
import numpy as np
import polars as pl
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from mir.biomarkers.metaclonotype_cluster import (
    MetaclonotypeClusterConfig,
    cluster_metaclonotypes,
    cluster_paired_metaclonotypes,
)
from mir.common.metaclonotype import (
    functional_diversity,
    functional_hill_curve,
    summarize_metaclonotypes,
)
from mir.common.diversity import summarize_clonotypes, hill_curve_clonotypes
from mir.utils import ensure_airr_benchmark

In [ ]:
# Cell 3: Load benchmark data
from mir.common.parser import AdaptiveParser
from mir.utils import ensure_airr_benchmark, find_airr_benchmark_sra_meta

ensure_airr_benchmark()

meta_df = find_airr_benchmark_sra_meta()
print(meta_df.head())

In [ ]:
# Cell 4: Parse a single TRB repertoire for single-chain comparisons
from pathlib import Path
from mir.utils import notebook_large_assets_root

# Pick first available sample
airr_root = notebook_large_assets_root() / 'airr_benchmark'
sample_files = sorted(airr_root.glob('*.tsv.gz'))[:1]
assert sample_files, 'No sample files found — run ensure_airr_benchmark() first'

parser = AdaptiveParser(locus='TRB')
rep = parser.parse_file(str(sample_files[0]))
print(f'Loaded {len(rep.clonotypes)} clonotypes from {sample_files[0].name}')

In [ ]:
# Cell 5: Baseline clonotype diversity
div_baseline = summarize_clonotypes(rep.clonotypes)
print('Baseline diversity:')
for field in ['abundance', 'diversity', 'shannon', 'gini_simpson', 'chao1']:
    print(f'  {field}: {getattr(div_baseline, field):.3f}')

In [ ]:
# Cell 6: Edit-distance graph metaclonotypes (components and Leiden)
results = {}

for graph_algo in ('components', 'leiden'):
    cfg = MetaclonotypeClusterConfig(
        method='edit_distance',
        metric='hamming',
        threshold=1,
        graph_algo=graph_algo,
        min_cluster_size=2,
        n_jobs=4,
    )
    t0 = time.perf_counter()
    meta = cluster_metaclonotypes(rep, cfg)
    elapsed = time.perf_counter() - t0
    div = functional_diversity(rep, meta)
    results[f'edit_distance/{graph_algo}'] = (meta, div, elapsed)
    print(f'edit_distance/{graph_algo}: clusters={meta.n_clusters}, '
          f'shannon={div.shannon:.3f}, elapsed={elapsed:.2f}s')

In [ ]:
# Cell 7: ALICE metaclonotypes (requires ALICE to be pre-run)
# Run ALICE enrichment first, then build metaclonotypes.
from mir.biomarkers.alice import add_alice_metadata

t0 = time.perf_counter()
rep_alice = add_alice_metadata(
    rep,
    species='human',
    metric='hamming',
    threshold=1,
    match_mode='vj',
)
t_alice_analysis = time.perf_counter() - t0

cfg_alice = MetaclonotypeClusterConfig(method='alice', q_value_max=0.05)
t0 = time.perf_counter()
meta_alice = cluster_metaclonotypes(rep_alice, cfg_alice)
t_alice_cluster = time.perf_counter() - t0

div_alice = functional_diversity(rep_alice, meta_alice)
results['alice'] = (meta_alice, div_alice, t_alice_analysis + t_alice_cluster)
print(f'ALICE: clusters={meta_alice.n_clusters}, '
      f'shannon={div_alice.shannon:.3f}, '
      f'analysis={t_alice_analysis:.2f}s, cluster={t_alice_cluster:.3f}s')

In [ ]:
# Cell 8: TCRdist metaclonotypes
cfg_tcrdist = MetaclonotypeClusterConfig(
    method='tcrdist',
    locus='TRB',
    species='human',
    max_distance=24.5,
    n_jobs=4,
)
t0 = time.perf_counter()
meta_tcrdist = cluster_metaclonotypes(rep, cfg_tcrdist)
elapsed = time.perf_counter() - t0

div_tcrdist = functional_diversity(rep, meta_tcrdist)
results['tcrdist'] = (meta_tcrdist, div_tcrdist, elapsed)
print(f'TCRdist: clusters={meta_tcrdist.n_clusters}, '
      f'shannon={div_tcrdist.shannon:.3f}, elapsed={elapsed:.2f}s')

In [ ]:
# Cell 9: TCREmp single-chain metaclonotypes
cfg_tcremp = MetaclonotypeClusterConfig(
    method='tcremp',
    locus='TRB',
    species='human',
    n_prototypes=300,
    embed_cluster_algo='dbscan',
    dbscan_eps=0.5,
    dbscan_min_samples=3,
    n_jobs=4,
)
t0 = time.perf_counter()
meta_tcremp = cluster_metaclonotypes(rep, cfg_tcremp)
elapsed = time.perf_counter() - t0

div_tcremp = functional_diversity(rep, meta_tcremp)
results['tcremp'] = (meta_tcremp, div_tcremp, elapsed)
print(f'TCREmp DBSCAN: clusters={meta_tcremp.n_clusters}, '
      f'shannon={div_tcremp.shannon:.3f}, elapsed={elapsed:.2f}s')

In [ ]:
# Cell 10: Summary table — single-chain comparisons
rows = []
for method, (meta, div, elapsed) in results.items():
    rows.append({
        'method': method,
        'n_clusters': meta.n_clusters,
        'shannon': round(div.shannon, 3),
        'gini_simpson': round(div.gini_simpson, 3),
        'chao1': round(div.chao1, 1),
        'elapsed_s': round(elapsed, 2),
    })

# Baseline row
rows.insert(0, {
    'method': 'baseline (clonotypes)',
    'n_clusters': div_baseline.abundance,
    'shannon': round(div_baseline.shannon, 3),
    'gini_simpson': round(div_baseline.gini_simpson, 3),
    'chao1': round(div_baseline.chao1, 1),
    'elapsed_s': 0.0,
})

summary_df = pl.DataFrame(rows)
print(summary_df)

In [ ]:
# Cell 11: Hill curves comparison across methods
q_values = [0.0, 0.5, 1.0, 1.5, 2.0, 3.0]

fig, ax = plt.subplots(figsize=(7, 4))

# Baseline Hill curve
hill_base = hill_curve_clonotypes(rep.clonotypes, q_values=q_values)
ax.plot(hill_base['q'], hill_base['D_q'], 'k--', lw=1.5, label='baseline (clonotypes)')

colors = plt.cm.tab10.colors
for i, (method, (meta, _div, _)) in enumerate(results.items()):
    # Use the rep that was used for that method (alice uses rep_alice)
    r = rep_alice if method == 'alice' else rep
    try:
        hill = functional_hill_curve(r, meta, q_values=q_values)
        ax.plot(hill['q'], hill['D_q'], color=colors[i % 10],
                lw=2, label=method)
    except Exception as exc:
        print(f'  Hill curve for {method} skipped: {exc}')

ax.set_xlabel('Hill order q', fontsize=12)
ax.set_ylabel('Functional diversity D(q)', fontsize=12)
ax.set_title('Functional Hill curves by clustering method', fontsize=13)
ax.legend(fontsize=9, loc='upper right')
ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

---
## Paired-chain analysis

For single-cell paired TRA/TRB data we can:
1. Use TCREmp's **native paired embedding** (`PairedTCREmp`) — joint representation of both chains.
2. Run **any single-chain method independently on each chain** then combine cluster IDs via
   `cluster_paired_metaclonotypes` → cluster ID = `"{TRA_cluster}.{TRB_cluster}"`.

Below we compare TCREmp-native paired vs TCRNET-combined on the same paired repertoire.

In [ ]:
# Cell 12: Load paired TRA/TRB 10x data
from mir.utils import find_airr_benchmark_dcode_10x_vdj_v1_donor
from mir.common.single_cell import SingleCellRepertoire
from mir.common.single_cell_parser import load_10x_vdj_v1_cell_clonotypes

donor_path = find_airr_benchmark_dcode_10x_vdj_v1_donor()
if donor_path is None:
    print('10x VDJ donor data not available — skipping paired analysis.')
    paired_rep = None
else:
    sc_rep = load_10x_vdj_v1_cell_clonotypes(str(donor_path))
    paired_reps = sc_rep.paired_repertoire.paired_locus_repertoires
    tra_trb = paired_reps.get('TRA_TRB')
    if tra_trb:
        print(f'Loaded {len(tra_trb.paired_clonotypes)} TRA/TRB pairs')
        paired_rep = tra_trb
    else:
        print('No TRA_TRB pairs found.')
        paired_rep = None

In [ ]:
# Cell 13: Single-chain-combined paired metaclonotypes (edit-distance)
if paired_rep is not None:
    cfg_ed_paired = MetaclonotypeClusterConfig(
        method='edit_distance',
        metric='hamming',
        threshold=1,
        graph_algo='components',
        min_cluster_size=1,
        n_jobs=4,
    )
    t0 = time.perf_counter()
    meta_ed_paired = cluster_paired_metaclonotypes(paired_rep, cfg_ed_paired)
    t_ed = time.perf_counter() - t0
    print(f'Edit-distance combined: paired_clusters={meta_ed_paired.n_clusters}, '
          f'elapsed={t_ed:.2f}s')
else:
    print('Skipped: no paired repertoire.')

In [ ]:
# Cell 14: Native paired TCREmp metaclonotypes
if paired_rep is not None:
    cfg_tcremp_paired = MetaclonotypeClusterConfig(
        method='tcremp',
        locus_pair='TRA_TRB',
        species='human',
        n_prototypes=300,
        embed_cluster_algo='dbscan',
        dbscan_eps=0.5,
        dbscan_min_samples=3,
        n_jobs=4,
    )
    t0 = time.perf_counter()
    # cluster_paired_metaclonotypes with method=tcremp uses PairedTCREmp natively
    meta_tcremp_paired = cluster_paired_metaclonotypes(paired_rep, cfg_tcremp_paired)
    t_tcremp = time.perf_counter() - t0
    print(f'TCREmp native paired: paired_clusters={meta_tcremp_paired.n_clusters}, '
          f'elapsed={t_tcremp:.2f}s')
else:
    print('Skipped: no paired repertoire.')

In [ ]:
# Cell 15: Concordance between TCREmp-native and edit-distance-combined
# Two paired metaclonotype sets are concordant if pairs assigned to the same
# cluster by method A are also largely co-clustered by method B.

if paired_rep is not None and 'meta_tcremp_paired' in dir():
    ed_t = meta_ed_paired.table.rename({'cluster_id': 'cluster_ed'})
    tcp_t = meta_tcremp_paired.table.rename({'cluster_id': 'cluster_tcremp'})

    joined = ed_t.join(
        tcp_t,
        on=['clonotype_id_1', 'clonotype_id_2'],
        how='inner',
    )

    # Cross-table: count pairs per (ed_cluster, tcremp_cluster)
    cross = (
        joined
        .group_by(['cluster_ed', 'cluster_tcremp'])
        .len()
        .sort('len', descending=True)
    )
    print(f'Pairs in both tables: {len(joined)}')
    print(f'Top 10 concordant cluster pairs:')
    print(cross.head(10))

    # Adjusted Rand Index for cluster agreement
    from sklearn.metrics import adjusted_rand_score

    # Create integer labels from cluster IDs for shared pairs
    ed_labels = joined['cluster_ed'].cast(pl.Categorical).to_physical().to_list()
    tcp_labels = joined['cluster_tcremp'].cast(pl.Categorical).to_physical().to_list()
    ari = adjusted_rand_score(ed_labels, tcp_labels)
    print(f'\nAdjusted Rand Index (edit_distance vs TCREmp): {ari:.4f}')
else:
    print('Skipped: paired reps not available.')

In [ ]:
# Cell 16: Cluster size distribution comparison (paired methods)
if paired_rep is not None and 'meta_tcremp_paired' in dir():
    fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=False)
    for ax, (meta, label) in zip(
        axes,
        [(meta_ed_paired, 'Edit-distance combined'), (meta_tcremp_paired, 'TCREmp native paired')]
    ):
        sizes = (
            meta.table
            .group_by('cluster_id')
            .len()
            ['len']
            .to_list()
        )
        ax.hist(sizes, bins=30, edgecolor='white', color='steelblue', alpha=0.85)
        ax.set_title(label, fontsize=12)
        ax.set_xlabel('Pairs per cluster', fontsize=11)
        ax.set_ylabel('Number of clusters', fontsize=11)
        ax.set_yscale('log')
        ax.grid(alpha=0.3)

    fig.suptitle('Paired metaclonotype cluster size distributions', fontsize=13)
    fig.tight_layout()
    plt.show()
else:
    print('Skipped: paired reps not available.')

---
## Key takeaways

- **Edit-distance (components)** is fastest and requires no pre-trained model.
- **ALICE / TCRNET** restrict clusters to statistically enriched seeds, giving fewer but more targeted clusters.
- **TCRdist** uses continuous alignment scores and V-gene germline information.
- **TCREmp** captures global V/J/CDR3 similarity via learned prototype embeddings.
- **Paired from single-chain** (`cluster_paired_metaclonotypes`) applies any single-chain method per chain then combines IDs, making paired-chain analysis available for *all* methods.
- **TCREmp native paired** uses a joint embedding of both chains, which can capture TRA/TRB co-variation not visible in chain-independent methods.
- Concordance (ARI) between strategies reflects how similar their notions of clonotype similarity are.